# DATA 266 HW2.5 — GPU Assignment I: Precision, Bandwidth, and the Cost of Attention

Single self-contained notebook: no separate `.py` modules or shell scripts.
Run this top to bottom **on the GPU workstation** (a CUDA-enabled machine with
PyTorch installed) to produce every result. It was authored and reviewed on a
Mac with no NVIDIA GPU, so it has **not been executed** and contains no cell
outputs.

**GPU note:** the assignment specifies a GPU Lab RTX 5090 or RTX 4090
workstation. This run targets a personal **RTX 4060** instead. The vendor
spec cell in Part A is a single dictionary you edit to match whichever card
you actually use — update it before trusting any "% of theoretical peak"
number if you end up running this on a different card.

Do not fabricate outputs by hand-editing this notebook. Run the cells for
real, then transcribe the observed values into `reports/METRICS.md` and
`reports/RUN_LOG.txt`.


In [ ]:
# Personal parameters (SID4 = last four digits of SJSU ID 019089486)
SID4 = 9486
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10
print(f"SID4={SID4}, SEED={SEED}, SLICE={SLICE}, HP_ID={HP_ID}, CLS_A={CLS_A}, CLS_B={CLS_B}")
print("HW2.5 has no HP_ID mapping and uses no second model/hyperparameter arm.")


## Setup

Imports, output directories, and GPU verification. Everything below assumes
this cell has run successfully.

In [ ]:
import csv
import json
import math
import subprocess
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

REPO_ROOT = Path.cwd()
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
for sub in ["system_info", "precision", "bandwidth", "attention", "thermal"]:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA available:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("Device name:", torch.cuda.get_device_name(0))
else:
    print("No CUDA device detected. Run this notebook on the GPU workstation.")


In [ ]:
def get_gpu_uuid():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    )
    return out.stdout.strip().splitlines()[0]

GPU_UUID = get_gpu_uuid() if CUDA_AVAILABLE else None
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
print("GPU UUID:", GPU_UUID)
print("GPU name:", GPU_NAME)


## Part A — Onboarding and Provenance

Captures the complete `nvidia-smi -q` output, measured hardware facts (UUID,
driver, CUDA version, VRAM, power limit), and vendor-documented specs for the
card actually used. Measured and vendor-documented values are kept separate
on purpose: measured values come from `nvidia-smi`/`torch` at run time,
vendor values come from NVIDIA's published documentation and are edited by
hand below.

In [ ]:
# Full nvidia-smi -q capture. If you run this on more than one machine
# (e.g. a lab RTX 4090 in addition to your RTX 4060), re-run this cell there
# and save it under a distinct filename before it gets overwritten.
smi_q = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True, check=True)
smi_path = RESULTS_DIR / "system_info" / "nvidia_smi_full_query.txt"
smi_path.write_text(smi_q.stdout)
print("Saved", smi_path)


In [ ]:
# Measured hardware facts -- queried live, never hardcoded.
fields = "uuid,name,driver_version,memory.total,power.limit,power.max_limit,compute_cap"
q = subprocess.run(
    ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
)
values = [v.strip() for v in q.stdout.strip().splitlines()[0].split(",")]
measured_hardware_info = dict(zip(fields.split(","), values))
measured_hardware_info["captured_at_utc"] = datetime.now(timezone.utc).isoformat()
measured_hardware_info["torch_version"] = torch.__version__
measured_hardware_info["torch_cuda_version"] = torch.version.cuda

measured_path = RESULTS_DIR / "system_info" / "measured_hardware_info.json"
measured_path.write_text(json.dumps(measured_hardware_info, indent=2))
measured_hardware_info


In [ ]:
# VENDOR-DOCUMENTED SPECIFICATIONS -- NOT measured. Edit this dict by hand to
# match whichever card `measured_hardware_info['name']` actually reports,
# using NVIDIA's official spec page and the Ada/Blackwell whitepaper as your
# source. The values below are pre-filled for an RTX 4060 (8 GB); if you run
# this on the assignment's specified RTX 4090 or RTX 5090 lab machine instead,
# replace every field here and re-run the cells below it.
#
# The RTX 4060 TF32/FP16/BF16 theoretical peak TFLOPS were derived from
# NVIDIA's published FP32 figure and RTX 4090 Tensor Core ratios (not read
# directly off a single vendor table) because outbound network access to
# nvidia.com was unavailable while authoring this notebook -- verify all four
# figures against the live NVIDIA page before trusting Part B's "% of peak"
# column.
VENDOR_SPECS = {
    "gpu_name": "NVIDIA GeForce RTX 4060",
    "architecture": "Ada Lovelace (AD107)",
    "memory_type": "GDDR6",
    "memory_bandwidth_gbps_spec": 272.0,
    "vram_capacity_gb_spec": 8,
    "power_limit_w_spec": 115,
    "tensor_core_generation": "4th generation (Ada)",
    "reduced_precisions_supported": ["FP16", "BF16", "TF32", "INT8", "INT4", "FP8"],
    "theoretical_peak_tflops": {"fp32": 15.11, "tf32": 15.11, "fp16": 60.45, "bf16": 30.2},
    "vendor_documentation_citation": "https://www.nvidia.com/en-us/geforce/graphics-cards/40-series/rtx-4060-4060ti/",
}

vendor_path = RESULTS_DIR / "system_info" / "vendor_specs.json"
vendor_path.write_text(json.dumps(VENDOR_SPECS, indent=2))
print(f"GPU UUID for this run: {GPU_UUID}")
print("Record this UUID in provenance/reservation_record.md and reports/RUN_LOG.txt.")
VENDOR_SPECS


## Part B — Precision and Achieved Throughput

Dense square matmul at N = 1024, 4096, 8192, 16384 for FP32, TF32, FP16, and
BF16. Every run warms up, synchronizes CUDA before/after timing, repeats for
a recorded number of repetitions, and reports achieved TFLOPS plus percent of
`VENDOR_SPECS`'s theoretical peak for that precision. A CUDA out-of-memory
error is caught per (N, precision) pair instead of aborting the sweep.

In [ ]:
MATRIX_SIZES = [1024, 4096, 8192, 16384]
PRECISIONS = ["fp32", "tf32", "fp16", "bf16"]
REPS_BY_SIZE = {1024: 50, 4096: 30, 8192: 15, 16384: 8}
WARMUP_BY_SIZE = {1024: 10, 4096: 8, 8192: 5, 16384: 3}


def dtype_for(precision):
    return {
        "fp32": torch.float32, "tf32": torch.float32,
        "fp16": torch.float16, "bf16": torch.bfloat16,
    }[precision]


def benchmark_matmul(n, precision):
    torch.backends.cuda.matmul.allow_tf32 = (precision == "tf32")
    dtype = dtype_for(precision)
    reps, warmup = REPS_BY_SIZE[n], WARMUP_BY_SIZE[n]
    row = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "gpu_name": GPU_NAME, "precision": precision, "matrix_n": n,
        "warmup_iters": warmup, "repetitions": reps, "status": "success", "error_message": "",
    }
    try:
        torch.cuda.empty_cache()
        a = torch.randn(n, n, device="cuda", dtype=dtype)
        b = torch.randn(n, n, device="cuda", dtype=dtype)
        for _ in range(warmup):
            _ = a @ b
        torch.cuda.synchronize()

        latencies_ms = []
        for _ in range(reps):
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = a @ b
            torch.cuda.synchronize()
            latencies_ms.append((time.perf_counter() - start) * 1000.0)

        mean_ms = sum(latencies_ms) / len(latencies_ms)
        flops = 2 * (n ** 3)
        tflops = flops / (mean_ms / 1000.0) / 1e12
        peak = VENDOR_SPECS["theoretical_peak_tflops"][precision]
        row.update({
            "mean_latency_ms": mean_ms,
            "achieved_tflops": tflops,
            "theoretical_peak_tflops": peak,
            "pct_of_theoretical_peak": tflops / peak * 100.0 if peak else float("nan"),
        })
        del a, b
    except torch.cuda.OutOfMemoryError as exc:
        row.update({"status": "oom", "error_message": f"{type(exc).__name__}: {exc}"})
    finally:
        torch.cuda.empty_cache()
    return row


precision_rows = [benchmark_matmul(n, p) for n in MATRIX_SIZES for p in PRECISIONS]
precision_df = pd.DataFrame(precision_rows)
precision_df.to_csv(RESULTS_DIR / "precision" / "precision_benchmark_results.csv", index=False)
precision_df


In [ ]:
# Additional lower-precision probe (FP8). RTX 40-series Tensor Cores support
# FP8, but PyTorch only exposes FP8 GEMM through torch._scaled_mm, which is
# version/build-dependent. This records a real measurement or the exact
# unavailability message -- it never pretends FP8 worked if it did not.
def attempt_fp8_probe(n=4096, reps=20, warmup=5):
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "precision_attempted": "fp8_e4m3", "api_attempted": "torch._scaled_mm",
    }
    try:
        if not hasattr(torch, "float8_e4m3fn") or not hasattr(torch, "_scaled_mm"):
            raise RuntimeError(
                "torch.float8_e4m3fn or torch._scaled_mm is not available in this "
                "PyTorch build; FP8 GEMM is not exposed by the installed software stack."
            )
        a = torch.randn(n, n, device="cuda", dtype=torch.float32).to(torch.float8_e4m3fn)
        b = torch.randn(n, n, device="cuda", dtype=torch.float32).to(torch.float8_e4m3fn)
        scale = torch.tensor(1.0, device="cuda")
        for _ in range(warmup):
            _ = torch._scaled_mm(a, b.t(), scale_a=scale, scale_b=scale, out_dtype=torch.bfloat16)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(reps):
            _ = torch._scaled_mm(a, b.t(), scale_a=scale, scale_b=scale, out_dtype=torch.bfloat16)
        torch.cuda.synchronize()
        mean_ms = (time.perf_counter() - start) * 1000.0 / reps
        tflops = 2 * (n ** 3) / (mean_ms / 1000.0) / 1e12
        record.update({"status": "available", "mean_latency_ms": mean_ms, "achieved_tflops": tflops})
    except Exception as exc:
        record.update({"status": "unavailable", "error_type": type(exc).__name__, "error_message": str(exc)})
    return record


fp8_result = attempt_fp8_probe()
(RESULTS_DIR / "precision" / "fp8_availability_probe.json").write_text(json.dumps(fp8_result, indent=2))
fp8_result


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
successful = precision_df[precision_df.status == "success"]
for precision, group in successful.groupby("precision"):
    group = group.sort_values("matrix_n")
    ax.plot(group.matrix_n, group.achieved_tflops, marker="o", label=precision.upper())
ax.set_xscale("log", base=2)
ax.set_xlabel("Matrix size N (N x N)")
ax.set_ylabel("Achieved TFLOPS")
ax.set_title("Part B — Achieved TFLOPS vs. Matrix Size by Precision")
ax.legend(title="Precision")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "precision_tflops_vs_matrix_size.png", dpi=200)
plt.show()


**Plateau discussion (fill in after running the cells above):** for each
precision, state the matrix size at which achieved TFLOPS plateaus and,
using the actual `precision_df` data, explain why smaller matrices fall
short of peak throughput (fixed kernel-launch overhead, too little work to
saturate all streaming multiprocessors, or a memory-bound regime at small N
per the Part C roofline). Do not state a plateau size before the benchmark
has actually run.

## Part C — Bandwidth-Bound vs. Compute-Bound

A memory-bound elementwise addition and a compute-bound square matmul,
compared against `VENDOR_SPECS`'s roofline ridge point (theoretical peak
FP32 FLOPS / theoretical peak bandwidth). The elementwise-add tensor size
adaptively shrinks and retries on a CUDA out-of-memory error, which matters
more on an 8 GB card than a 24 GB one.

In [ ]:
def roofline_ridge_point():
    peak_flops = VENDOR_SPECS["theoretical_peak_tflops"]["fp32"] * 1e12
    peak_bytes_per_s = VENDOR_SPECS["memory_bandwidth_gbps_spec"] * 1e9
    return peak_flops / peak_bytes_per_s


def classify(intensity, ridge_point):
    return "compute_bound" if intensity >= ridge_point else "bandwidth_bound"


def benchmark_elementwise_add(target_n=200_000_000, min_n=12_500_000, reps=20, warmup=5):
    n = target_n
    while n >= min_n:
        try:
            torch.cuda.empty_cache()
            a = torch.randn(n, device="cuda", dtype=torch.float32)
            b = torch.randn(n, device="cuda", dtype=torch.float32)
            for _ in range(warmup):
                c = a + b
            torch.cuda.synchronize()

            latencies_ms = []
            for _ in range(reps):
                torch.cuda.synchronize()
                start = time.perf_counter()
                c = a + b
                torch.cuda.synchronize()
                latencies_ms.append((time.perf_counter() - start) * 1000.0)

            mean_ms = sum(latencies_ms) / len(latencies_ms)
            bytes_moved = 3 * n * 4  # read a, read b, write c (float32)
            flops = n
            gbps = (bytes_moved / (mean_ms / 1000.0)) / 1e9
            pct = (gbps / VENDOR_SPECS["memory_bandwidth_gbps_spec"]) * 100.0
            intensity = flops / bytes_moved
            ridge = roofline_ridge_point()
            del a, b, c
            torch.cuda.empty_cache()
            return {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
                "operation": "elementwise_add", "category": "memory_bound", "n": n,
                "status": "success", "mean_latency_ms": mean_ms, "achieved_gbps": gbps,
                "pct_of_spec_bandwidth": pct, "arithmetic_intensity_flops_per_byte": intensity,
                "roofline_ridge_point_flops_per_byte": ridge, "roofline_classification": classify(intensity, ridge),
            }
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            n //= 2
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "operation": "elementwise_add", "category": "memory_bound", "n": None,
        "status": "oom", "note": f"Did not fit even at the minimum size ({min_n} elements).",
    }


def benchmark_compute_bound_matmul(n=8192, reps=15, warmup=5):
    torch.backends.cuda.matmul.allow_tf32 = False  # clean FP32 reference point
    a = torch.randn(n, n, device="cuda", dtype=torch.float32)
    b = torch.randn(n, n, device="cuda", dtype=torch.float32)
    for _ in range(warmup):
        _ = a @ b
    torch.cuda.synchronize()

    latencies_ms = []
    for _ in range(reps):
        torch.cuda.synchronize()
        start = time.perf_counter()
        _ = a @ b
        torch.cuda.synchronize()
        latencies_ms.append((time.perf_counter() - start) * 1000.0)

    mean_ms = sum(latencies_ms) / len(latencies_ms)
    flops = 2 * (n ** 3)
    bytes_moved = 3 * (n ** 2) * 4  # read A, read B, write C (naive lower bound)
    gbps = (bytes_moved / (mean_ms / 1000.0)) / 1e9
    intensity = flops / bytes_moved
    ridge = roofline_ridge_point()
    del a, b
    torch.cuda.empty_cache()
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "operation": "square_matmul", "category": "compute_bound", "n": n,
        "status": "success", "mean_latency_ms": mean_ms, "achieved_gbps": gbps,
        "arithmetic_intensity_flops_per_byte": intensity,
        "roofline_ridge_point_flops_per_byte": ridge, "roofline_classification": classify(intensity, ridge),
    }


bandwidth_rows = [benchmark_elementwise_add(), benchmark_compute_bound_matmul()]
bandwidth_df = pd.DataFrame(bandwidth_rows)
bandwidth_df.to_csv(RESULTS_DIR / "bandwidth" / "bandwidth_benchmark_results.csv", index=False)
bandwidth_df


## Part D — Cost of Attention

Naive scaled dot-product attention (materializes the full
`[seq_len, seq_len]` score/probability matrix) vs. PyTorch's fused
`scaled_dot_product_attention`. Fixed configuration: batch size 1, 1
attention head, head dimension 64, dtype bfloat16, under
`torch.inference_mode()`. CUDA out-of-memory errors are caught per sequence
length so one failure does not stop the sweep; after the fixed grid, the
boundary between the largest success and smallest failure is refined with a
few extra probes -- the exact single-token failure point is never claimed.

In [ ]:
BATCH_SIZE, NUM_HEADS, HEAD_DIM = 1, 1, 64
ATTENTION_DTYPE = torch.bfloat16
SEQ_LENGTHS = [512, 1024, 2048, 4096, 8192, 16384]
ATTENTION_REPS, ATTENTION_WARMUP = 10, 3
MAX_BOUNDARY_PROBES = 4


def naive_attention(q, k, v):
    d_k = q.shape[-1]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return weights @ v


def fused_attention(q, k, v):
    try:
        from torch.nn.attention import SDPBackend, sdpa_kernel
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION]):
            return F.scaled_dot_product_attention(q, k, v)
    except ImportError:
        with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=True):
            return F.scaled_dot_product_attention(q, k, v)


def run_attention_once(seq_len, implementation):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    row = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu_uuid": GPU_UUID,
        "implementation": implementation, "dtype": "bfloat16", "batch_size": BATCH_SIZE,
        "num_heads": NUM_HEADS, "head_dim": HEAD_DIM, "seq_len": seq_len,
        "status": "success", "error_message": "",
    }
    fn = naive_attention if implementation == "naive" else fused_attention
    try:
        shape = (BATCH_SIZE, NUM_HEADS, seq_len, HEAD_DIM)
        q = torch.randn(shape, device="cuda", dtype=ATTENTION_DTYPE)
        k = torch.randn(shape, device="cuda", dtype=ATTENTION_DTYPE)
        v = torch.randn(shape, device="cuda", dtype=ATTENTION_DTYPE)
        with torch.inference_mode():
            for _ in range(ATTENTION_WARMUP):
                _ = fn(q, k, v)
            torch.cuda.synchronize()
            latencies_ms = []
            for _ in range(ATTENTION_REPS):
                torch.cuda.synchronize()
                start = time.perf_counter()
                _ = fn(q, k, v)
                torch.cuda.synchronize()
                latencies_ms.append((time.perf_counter() - start) * 1000.0)
        row["mean_latency_ms"] = sum(latencies_ms) / len(latencies_ms)
        row["peak_memory_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
        del q, k, v
    except torch.cuda.OutOfMemoryError as exc:
        row["status"] = "oom"
        row["error_message"] = f"{type(exc).__name__}: {exc}"
        row["peak_memory_mb"] = torch.cuda.max_memory_allocated() / (1024 ** 2)
    finally:
        torch.cuda.empty_cache()
    return row


attention_rows = []
for implementation in ("naive", "fused"):
    grid_rows = [run_attention_once(s, implementation) for s in SEQ_LENGTHS]
    attention_rows.extend(grid_rows)

    successes = sorted(r["seq_len"] for r in grid_rows if r["status"] == "success")
    failures = sorted(r["seq_len"] for r in grid_rows if r["status"] == "oom")
    if successes and failures and max(successes) < min(failures):
        low, high = max(successes), min(failures)
        for _ in range(MAX_BOUNDARY_PROBES):
            if high - low <= 64:
                break
            mid = ((low + high) // 2 // 64) * 64
            if mid <= low or mid >= high:
                break
            probe = run_attention_once(mid, implementation)
            attention_rows.append(probe)
            if probe["status"] == "success":
                low = mid
            else:
                high = mid

attention_df = pd.DataFrame(attention_rows)
attention_df.to_csv(RESULTS_DIR / "attention" / "attention_benchmark_results.csv", index=False)
attention_df


In [ ]:
def oom_boundary(df, implementation):
    subset = df[df.implementation == implementation]
    successes = subset.loc[subset.status == "success", "seq_len"]
    failures = subset.loc[subset.status == "oom", "seq_len"]
    return {
        "implementation": implementation,
        "largest_tested_success": int(successes.max()) if len(successes) else None,
        "smallest_tested_failure": int(failures.min()) if len(failures) else None,
        "note": "Bracket only; does not claim an exact single-token failure point.",
    }


naive_boundary = oom_boundary(attention_df, "naive")
fused_boundary = oom_boundary(attention_df, "fused")

naive_success = attention_df[
    (attention_df.implementation == "naive") & (attention_df.status == "success")
].sort_values("seq_len")
if len(naive_success) >= 3:
    a_coeff, b_coeff, c_coeff = np.polyfit(naive_success.seq_len, naive_success.peak_memory_mb, deg=2)
    memory_fit = {
        "status": "fit", "quadratic_coefficient_a": float(a_coeff),
        "linear_coefficient_b": float(b_coeff), "intercept_c": float(c_coeff),
        "note": "peak_memory_mb ~= a * seq_len^2 + b * seq_len + c",
    }
else:
    memory_fit = {"status": "insufficient_data", "note": "Need >= 3 successful naive runs to fit a quadratic."}

naive_latency = attention_df[
    (attention_df.implementation == "naive") & (attention_df.status == "success")
].set_index("seq_len")["mean_latency_ms"]
fused_latency = attention_df[
    (attention_df.implementation == "fused") & (attention_df.status == "success")
].set_index("seq_len")["mean_latency_ms"]
common_lengths = sorted(set(naive_latency.index) & set(fused_latency.index))
speedups = [
    {
        "seq_len": s, "naive_latency_ms": float(naive_latency[s]), "fused_latency_ms": float(fused_latency[s]),
        "speedup_naive_over_fused": float(naive_latency[s] / fused_latency[s]),
    }
    for s in common_lengths
]

attention_summary = {
    "gpu_uuid": GPU_UUID,
    "naive_oom_boundary": naive_boundary,
    "fused_oom_boundary": fused_boundary,
    "memory_quadratic_fit_naive": memory_fit,
    "speedups_fused_over_naive": speedups,
}
(RESULTS_DIR / "attention" / "attention_summary.json").write_text(json.dumps(attention_summary, indent=2))
attention_summary


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for implementation, marker in (("naive", "o"), ("fused", "s")):
    d = attention_df[
        (attention_df.implementation == implementation) & (attention_df.status == "success")
    ].sort_values("seq_len")
    if len(d):
        ax.plot(d.seq_len, d.peak_memory_mb, marker=marker, label=f"{implementation} (success)")
    failed = attention_df[
        (attention_df.implementation == implementation) & (attention_df.status == "oom")
    ].seq_len
    for x in failed:
        ax.axvline(x, color="red", linestyle=":", alpha=0.3)

if memory_fit.get("status") == "fit":
    xs_fit = np.linspace(naive_success.seq_len.min(), naive_success.seq_len.max(), 200)
    ys_fit = memory_fit["quadratic_coefficient_a"] * xs_fit ** 2 + memory_fit["linear_coefficient_b"] * xs_fit + memory_fit["intercept_c"]
    ax.plot(xs_fit, ys_fit, linestyle="--", color="black", alpha=0.6, label="naive quadratic fit")

ax.set_xlabel("Sequence length")
ax.set_ylabel("Peak allocated GPU memory (MB)")
ax.set_title("Part D — Peak Memory vs. Sequence Length (naive vs. fused attention)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "attention_peak_memory_vs_seqlen.png", dpi=200)
plt.show()


**What the fused kernel avoids materializing:** the fused
scaled-dot-product-attention backend (Flash/memory-efficient attention)
never materializes the full `[seq_len, seq_len]` attention probability
matrix in GPU memory. It processes queries, keys, and values in tiles,
computing partial attention scores and an online (running) softmax so only
small per-tile score buffers and running output/normalization statistics
stay resident. Because the `O(seq_len^2)` score and probability matrices --
and the extra GPU-memory read/write traffic they require -- are avoided
entirely, the fused kernel's peak memory scales close to `O(seq_len)`
instead of `O(seq_len^2)`, which is why it keeps succeeding at sequence
lengths where the naive implementation runs out of memory.

## Part E — Sustained Load and Thermal Behaviour

A ~20-minute sustained matmul load with a background thread sampling GPU
clocks, temperature, power draw, and utilization every 5 seconds. The CSV is
flushed after every row so the log survives an interruption. **This cell
blocks for the full duration** -- let it run to completion for the real
submission.

In [ ]:
def sample_gpu():
    fields = "uuid,clocks.sm,clocks.mem,temperature.gpu,power.draw,utilization.gpu"
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    )
    uuid, sm, mem, temp, power, util = [v.strip() for v in out.stdout.strip().splitlines()[0].split(",")]
    return {
        "gpu_uuid": uuid, "sm_clock_mhz": float(sm), "memory_clock_mhz": float(mem),
        "temperature_c": float(temp), "power_draw_w": float(power), "utilization_pct": float(util),
    }


CSV_FIELDS = [
    "elapsed_s", "timestamp_utc", "gpu_uuid", "sm_clock_mhz", "memory_clock_mhz",
    "temperature_c", "power_draw_w", "utilization_pct", "cumulative_matmuls",
    "interval_throughput_matmuls_per_s",
]


def run_sustained_load(duration_s=1200, interval_s=5.0, matmul_n=8192, output_csv=None):
    output_csv = output_csv or (RESULTS_DIR / "thermal" / "thermal_log.csv")
    a = torch.randn(matmul_n, matmul_n, device="cuda", dtype=torch.float16)
    b = torch.randn(matmul_n, matmul_n, device="cuda", dtype=torch.float16)
    counter = {"n": 0}
    stop_event = threading.Event()

    def sampler():
        start_time = time.time()
        last_count, last_sample_time = 0, start_time
        next_sample_at = start_time + interval_s
        with output_csv.open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
            writer.writeheader()
            f.flush()
            while not stop_event.is_set():
                now = time.time()
                if now < next_sample_at:
                    time.sleep(min(0.25, next_sample_at - now))
                    continue
                try:
                    sample = sample_gpu()
                except subprocess.CalledProcessError:
                    sample = {"gpu_uuid": "UNKNOWN", "sm_clock_mhz": float("nan"),
                              "memory_clock_mhz": float("nan"), "temperature_c": float("nan"),
                              "power_draw_w": float("nan"), "utilization_pct": float("nan")}
                now = time.time()
                current_count = counter["n"]
                dt = now - last_sample_time
                throughput = (current_count - last_count) / dt if dt > 0 else float("nan")
                row = {
                    "elapsed_s": round(now - start_time, 3),
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    **sample,
                    "cumulative_matmuls": current_count,
                    "interval_throughput_matmuls_per_s": throughput,
                }
                writer.writerow(row)
                f.flush()
                last_count, last_sample_time = current_count, now
                next_sample_at += interval_s

    sampler_thread = threading.Thread(target=sampler, daemon=True)
    sampler_thread.start()
    print(f"Starting sustained load: duration={duration_s}s, sampling every {interval_s}s, matmul_n={matmul_n}.")
    end_time = time.time() + duration_s
    try:
        while time.time() < end_time:
            _ = a @ b
            counter["n"] += 1
    except KeyboardInterrupt:
        print("Interrupted; saving partial log and stopping cleanly.")
    finally:
        torch.cuda.synchronize()
        stop_event.set()
        sampler_thread.join(timeout=interval_s * 2)
    print(f"Sustained load complete. Log saved continuously to {output_csv}")
    return output_csv


# ~20 minutes by default -- this is the real submission run, not a smoke test.
thermal_csv = run_sustained_load(duration_s=1200, interval_s=5.0)


In [ ]:
def analyze_thermal_log(csv_path):
    thermal_df = pd.read_csv(csv_path)
    if thermal_df.empty:
        return {"status": "no_data", "note": f"{csv_path} contains no rows yet."}

    total_duration = thermal_df.elapsed_s.max()
    first_30s = thermal_df[thermal_df.elapsed_s <= 30]
    last_5min = thermal_df[thermal_df.elapsed_s >= total_duration - 300]

    peak_clock = first_30s.sm_clock_mhz.max()
    threshold = 0.97 * peak_clock if pd.notna(peak_clock) else float("nan")
    throttle_onset_s = None
    consecutive_low = 0
    for _, row in thermal_df[thermal_df.elapsed_s > 30].iterrows():
        if pd.notna(row.sm_clock_mhz) and row.sm_clock_mhz <= threshold:
            consecutive_low += 1
            if consecutive_low >= 2 and throttle_onset_s is None:
                throttle_onset_s = row.elapsed_s
        else:
            consecutive_low = 0

    peak_throughput = first_30s.interval_throughput_matmuls_per_s.max()
    steady_throughput = last_5min.interval_throughput_matmuls_per_s.mean()
    steady_pct_of_peak = (
        steady_throughput / peak_throughput * 100.0
        if pd.notna(peak_throughput) and peak_throughput else float("nan")
    )

    return {
        "total_logged_duration_s": float(total_duration),
        "num_samples": len(thermal_df),
        "peak_sm_clock_mhz_first_30s": float(peak_clock) if pd.notna(peak_clock) else None,
        "max_temperature_c_observed": float(thermal_df.temperature_c.max()),
        "max_power_draw_w_observed": float(thermal_df.power_draw_w.max()),
        "throttling_detected": throttle_onset_s is not None,
        "throttle_onset_time_s": float(throttle_onset_s) if throttle_onset_s is not None else "none",
        "peak_throughput_matmuls_per_s_first_30s": float(peak_throughput) if pd.notna(peak_throughput) else None,
        "steady_state_throughput_matmuls_per_s_last_5min": float(steady_throughput) if pd.notna(steady_throughput) else None,
        "steady_state_pct_of_peak_throughput": float(steady_pct_of_peak) if pd.notna(steady_pct_of_peak) else None,
        "note": (
            "Throttling is flagged heuristically as the SM clock staying at or below "
            "97% of its first-30-second peak for two or more consecutive 5-second "
            "samples after the first 30 seconds. Confirm against the figure below "
            "before writing report conclusions."
        ),
    }


thermal_analysis = analyze_thermal_log(thermal_csv)
(RESULTS_DIR / "thermal" / "thermal_analysis.json").write_text(json.dumps(thermal_analysis, indent=2))
thermal_analysis


In [ ]:
thermal_df = pd.read_csv(thermal_csv)

fig, ax1 = plt.subplots(figsize=(9, 5.5))
ax1.plot(thermal_df.elapsed_s, thermal_df.sm_clock_mhz, color="tab:blue", label="SM clock (MHz)")
ax1.plot(thermal_df.elapsed_s, thermal_df.memory_clock_mhz, color="tab:cyan", label="Memory clock (MHz)")
ax1.set_xlabel("Elapsed time (s)")
ax1.set_ylabel("Clock (MHz)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(thermal_df.elapsed_s, thermal_df.temperature_c, color="tab:red", label="Temperature (C)")
ax2.set_ylabel("Temperature (C)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower center")
ax1.set_title("Part E — Clock and Temperature vs. Time (sustained load)")
ax1.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "thermal_clock_temperature_vs_time.png", dpi=200)
plt.show()


## Part F — Update the Reports

With every cell above executed for real on the GPU workstation:

1. Copy the observed values from `precision_df`, `bandwidth_df`,
   `attention_df`, `attention_summary`, and `thermal_analysis` (all also
   saved under `results/`) into `reports/METRICS.md`, replacing every
   "To be measured" placeholder, including Table HW2.5.1.
2. Append one entry per run to `reports/RUN_LOG.txt` using the template
   already in that file, so every value in `METRICS.md` is traceable to a
   UUID-labelled run.
3. Fill in `provenance/reservation_record.md` and `provenance/gpu_hours.md`
   from the actual GPU workstation reservation and session times.
4. Commit, push, and tag the completed assignment as `hw2-5` per the
   instructions in `README.md` -- only after the real results are in place.
